<div style="background:#0B3C49;color:white;padding:14px 18px;border-radius:8px">
<b style="color:#F2A541">INTERNATIONAL BALKAN UNIVERSITY · DS 501 INTRODUCTION TO DATA SCIENCE</b><br>
<span style="font-size:1.4em">Lab 2 · CRISP-DM Walkthrough: Telecom Churn</span><br>
Week 2 · Lesson 2-3 · about 60–75 minutes
</div>

## Learning goals
1. Apply all six CRISP-DM phases to one small project.
2. Write the business question and success criteria **before** touching the data.
3. Find and fix a real data-quality issue.
4. Evaluate a model against a **business cost**, not only accuracy.

**Dataset:** IBM Telco Customer Churn sample: 7,043 customers, their services, contract, charges, and whether they left last month (`Churn`). Modelling is deliberately simple; algorithms come in weeks 9–10.

# Phase 1 · Business understanding

**✏️ Your turn (written).** The retention team can call a limited number of customers each month with a discount. They ask: *which customers should we contact?* Complete: business objective; decision supported; analytic approach; target variable; business success criterion; model success criterion; one ethical risk.

*Your answer:* 

# Phase 2 · Data understanding

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
raw = pd.read_csv(URL)
print(raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
bad = raw[pd.to_numeric(raw["TotalCharges"], errors="coerce").isna()]
print(len(bad), "rows cannot be converted")
bad[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

**✏️ Your turn (written).** What do these rows have in common, and what value should `TotalCharges` take?

*Your answer:* 

In [ ]:
print(raw["Churn"].value_counts(normalize=True).round(3))
pd.crosstab(raw["Contract"], raw["Churn"], normalize="index").round(3) * 100

**✏️ Your turn.** Compute the churn rate (%) by `PaymentMethod` and by `InternetService`.

In [ ]:
# Hint: groupby the column and take the mean of (Churn == 'Yes')
# YOUR CODE HERE

**✏️ Your turn (written).** What should a retention manager take from these rates? Is it correlation or causation?

*Your answer:* 

# Phase 3 · Data preparation

In [ ]:
df = raw.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)
df = df.drop(columns=["customerID"])
df["Churn"] = (df["Churn"] == "Yes").astype(int)
cat_cols = df.drop(columns="Churn").select_dtypes(exclude="number").columns.tolist()
X = pd.get_dummies(df.drop(columns="Churn"), columns=cat_cols, drop_first=True, dtype=int)
y = df["Churn"]
print("Feature matrix:", X.shape)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print(X_train.shape, X_test.shape, round(y_train.mean(), 3), round(y_test.mean(), 3))

**✏️ Your turn (written).** Why split before fitting anything, and why `stratify=y`? Name one variable that would be leakage.

*Your answer:* 

# Phase 4 · Modelling

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
for name, m in [("Baseline", baseline), ("Logistic regression", logit)]:
    print(f"{name:20s} accuracy = {m.score(X_test, y_test):.3f}")

# Phase 5 · Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, roc_auc_score
proba = logit.predict_proba(X_test)[:, 1]
print("ROC AUC:", round(roc_auc_score(y_test, proba), 3))
def report(t):
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    return {"threshold": t, "contacted": tp + fp, "caught": tp, "missed": fn, "wasted": fp,
            "precision": round(precision_score(y_test, pred), 3), "recall": round(recall_score(y_test, pred), 3)}
pd.DataFrame([report(t) for t in (0.5, 0.3)])

### Evaluate against the business cost
Illustrative assumptions: an offer costs **€40**, a lost customer costs **€300** of future margin, and the offer keeps **50%** of contacted churners.

In [ ]:
OFFER_COST, LOST_VALUE, RETAIN_RATE = 40, 300, 0.5
def cost(t, retain=RETAIN_RATE):
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    return int((tp + fp) * OFFER_COST + (fn + tp * (1 - retain)) * LOST_VALUE)
thresholds = np.round(np.arange(0.05, 0.95, 0.05), 2)
costs = [cost(t) for t in thresholds]
best = thresholds[int(np.argmin(costs))]
plt.plot(thresholds, costs, marker="o"); plt.axvline(best, ls="--")
plt.xlabel("Decision threshold"); plt.ylabel("Total cost on test set (€)"); plt.title(f"Lowest cost at {best}"); plt.show()
print("Contact nobody:", int(y_test.sum()) * LOST_VALUE, "  Best threshold:", min(costs))

**✏️ Your turn.** Repeat the cost analysis with `RETAIN_RATE = 0.3`. Print the best threshold and its saving compared with contacting nobody.

In [ ]:
# Hint: call cost(t, retain=0.3) for each threshold
# YOUR CODE HERE

**✏️ Your turn (written).** Why is the cost-minimising threshold below 0.5, and which assumption matters most?

*Your answer:* 

# Phase 6 · Deployment and feedback

**✏️ Your turn (written).** In about 150 words: who uses the scores and when; what you would monitor; how you would collect feedback; and the ethics checkpoint.

*Your answer:* 

---
### Submission checklist
- [ ] All ✏️ cells completed
- [ ] `Kernel → Restart & Run All` runs without errors
- [ ] Notebook saved and uploaded to Moodle before the next class